In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd

# Configure Jupyter proxy
JupyterDash.infer_jupyter_proxy_config()

# Import CRUD module
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve all records
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId column
df.drop(columns=['_id'], inplace=True, errors='ignore')


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Header with logo linked to www.snhu.edu and unique identifier
    html.Center(html.A(
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 style={'height': '100px'}),
        href='https://www.snhu.edu', target='_blank'
    )),
    html.Center(html.B(html.H1('Grazioso Salvare Animal Shelter Dashboard'))),
    html.Center(html.P('Dashboard created by William Hijeregerman')),
    html.Hr(),

    # Radio buttons for rescue type filtering
    html.Div([
        html.Label('Select Rescue Type:', style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'water'},
                {'label': ' Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': ' Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': ' Reset', 'value': 'reset'}
            ],
            value='reset',
            inputStyle={'margin-right': '5px', 'margin-left': '15px'}
        )
    ]),
    html.Hr(),

    # Interactive data table with pagination and sorting
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        page_action='native',
        sort_action='native',
        sort_mode='multi',
        column_selectable='single',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'maxWidth': '200px',
                    'overflow': 'hidden', 'textOverflow': 'ellipsis'},
        style_header={'backgroundColor': 'rgb(230, 230, 230)', 'fontWeight': 'bold'}
    ),
    html.Br(),
    html.Hr(),

    # Charts displayed side-by-side: pie chart and geolocation map
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6'),
                 html.Div(id='map-id', className='col s12 m6')
             ])
])


#############################################
# Interaction Between Components / Controller
#############################################

# Callback to filter data table based on rescue type radio button selection
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'water':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'mountain':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'disaster':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    else:
        query = {}

    filtered_df = pd.DataFrame.from_records(db.read(query))
    filtered_df.drop(columns=['_id'], inplace=True, errors='ignore')
    return filtered_df.to_dict('records')


# Callback to display pie chart of breeds from current data table view
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Preferred Animals')
        )
    ]


# Callback to highlight a selected column in the data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Callback to update geolocation map based on selected row in data table
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if viewData is None:
        return []
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    # Austin TX is at [30.75, -97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tooltip and popup
            # Column 13 and 14 are location_lat and location_long
            # Column 4 is breed, Column 9 is name
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]], children=[
                dl.Tooltip(dff.iloc[row, 4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row, 9])
                ])
            ])
        ])
    ]


app.run_server(debug=True)
